# Bonus B4 solution, notebook 1: add a social media editor to the newsroom

The worked answer to the challenge of `b4a_meet_crewai.ipynb`. The setup repeats the notebook's tool and agents, so
this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
# CrewAI reads these when it is imported, so they come first
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"      # no usage statistics sent to CrewAI
os.environ["CREWAI_TRACING_ENABLED"] = "false"       # no traces sent to CrewAI's hosted platform
os.environ["CREWAI_DISABLE_VERSION_CHECK"] = "true"  # no check for a newer CrewAI on every start
os.environ["NO_COLOR"] = "1"                         # the crew's log prints without colours

# CrewAI's own files (the last run's task outputs) go to this lab's outputs/ folder, which git ignores
(Path.cwd() / "outputs").mkdir(exist_ok=True)
os.environ["CREWAI_STORAGE_DIR"] = str(Path.cwd() / "outputs")

In [3]:
import json
import re
from crewai import LLM, Agent, Task, Crew
from crewai.tools import tool

RESULT_SHEET = {
    "Northside Falcons": {
        "played": "Friday",
        "opponent": "Riverside Otters",
        "score": "Falcons 2, Otters 3",
        "falcons_goals": "Lina Haddad (2)",
        "winning_goal": "Otters, in minute 88",
    }
}


@tool("match_result")
def match_result(team: str) -> str:
    """Look up the result of a school team's latest match on the result sheet."""
    print(f"⚽ TOOL CALLED: match_result({team})")
    if team not in RESULT_SHEET:
        return f"no match on the result sheet for {team}"
    return json.dumps(RESULT_SHEET[team])


def new_llm():
    return LLM(model=f"openai/{MODEL}", api="responses", reasoning_effort=EFFORT, timeout=60)


def new_reporter():
    return Agent(
        role="Sports reporter",
        goal="Report school matches exactly as the result sheet records them.",
        backstory="You cover sport for The Northside Post, the paper of Northside School.",
        tools=[match_result],
        llm=new_llm(),
    )


def new_writer():
    return Agent(
        role="Headline writer",
        goal="Write short, lively headlines for the school paper.",
        backstory="You write the headlines of The Northside Post, the paper of Northside School. "
                  "You love the words football fans use.",
        llm=new_llm(),
    )


REPORT = "Report the latest match of the {team}: the opponent, the score and who scored."
HEADLINE = "Write the headline for the school paper's story about the latest match of the {team}."

## The editor and the post's guardrail

The editor has no tools, like the headline writer. Its task lists both earlier tasks in `context`, so its request
carries the report and the headline. `check_post` refuses a post that does not name Lina Haddad or runs over 280
characters, and prints each decision.

In [4]:
def new_editor():
    return Agent(
        role="Social media editor",
        goal="Write one short post about each match for the school paper's social media page.",
        backstory="You run the social media page of The Northside Post, the paper of Northside School.",
        llm=new_llm(),
    )


def check_post(output):
    post = output.raw
    if "Lina Haddad" not in post:
        print("❌ refused (no Lina Haddad):", post)
        return (False, "Name the scorer in full: Lina Haddad.")
    if len(post) > 280:
        print(f"❌ refused ({len(post)} characters):", post)
        return (False, "A post is at most 280 characters.")
    print(f"✅ accepted ({len(post)} characters)")
    return (True, post)

## Three tasks in one crew

In [5]:
reporter, writer, editor = new_reporter(), new_writer(), new_editor()

report_task = Task(description=REPORT, expected_output="Two or three plain sentences.", agent=reporter)
headline_task = Task(description=HEADLINE, expected_output="One headline, on one line.", agent=writer,
                     context=[report_task])
post_task = Task(description="Write one social media post about the latest match of the {team}.",
                 expected_output="One post, at most 280 characters.", agent=editor,
                 context=[report_task, headline_task], guardrail=check_post)

result = await Crew(agents=[reporter, writer, editor], tasks=[report_task, headline_task, post_task]).akickoff(
    inputs={"team": "Northside Falcons"})

print()
for output in result.tasks_output:
    print(f"{output.agent}: {output.raw}")
print()
print("model requests:", result.token_usage.successful_requests)

⚽ TOOL CALLED: match_result(Northside Falcons)


✅ accepted (103 characters)

Sports reporter: The Northside Falcons lost 3–2 to the Riverside Otters. Lina Haddad scored both goals for the Falcons.
Headline writer: Haddad’s Brace Not Enough as Falcons Fall 3–2 to Otters
Social media editor: Lina Haddad scored twice, but the Northside Falcons fell 3–2 to the Riverside Otters. #NorthsideFalcons

model requests: 4


Three task outputs. The editor's post, *Lina Haddad scored twice, but the Northside Falcons fell 3–2 to the Riverside
Otters. #NorthsideFalcons*, names the scorer and carries the score, and `check_post` accepted it at the first attempt
(103 characters). **4 model requests for three tasks**: two for the reporter, one each for the writer and the editor.
The headline writer still says *Brace*: this crew has no headline guardrail.

## What the editor was given

In [6]:
print(post_task.prompt_context)

The Northside Falcons lost 3–2 to the Riverside Otters. Lina Haddad scored both goals for the Falcons.

----------

Haddad’s Brace Not Enough as Falcons Fall 3–2 to Otters


The context is the two earlier outputs in the order the list gives them, the report first and the headline second,
with a line of dashes between them. The editor's request carries both, and nothing from the result sheet itself: its
facts come from the reporter, as the headline writer's do.